In [1]:
import pandas as pd
import functions as f
import numpy as np
import requests
from PIL import Image
from io import BytesIO
from textblob import TextBlob
import nltk
from nltk.corpus import stopwords
import ast
from bertopic import BERTopic

In [2]:
df_temporal = pd.read_csv('data/df_temporal.csv')
rating_temporal = pd.read_csv('data/rating_temporal.csv')

# DATETIME: Calculando sobre el tiempo

In [3]:
## Explorando nuestro df nos damos cuenta de que tiempo no ha sido correctamente unificado, así que unificamos. 

#### La columna published_date: 

In [4]:
## En este caso el problema es que las fechas de publicación tienen formatos dispares. 
# A veces solo tenemos el año y otras tenemos mes y año y otros caracteres. 
# Además, cada año puede estar al principio o al final, por lo que no podemos eliminar por posición. 

In [5]:
df_temporal.columns

Index(['title', 'description', 'authors', 'image', 'publisher',
       'published_date', 'categories', 'ratings_count', 'color_rgb'],
      dtype='object')

In [6]:
df_temporal['published_date'].head(20)

0           1996
1     2012-08-21
2     2018-09-20
3        2002-11
4           2005
5     2003-10-10
6     1981-09-01
7           1984
8     1992-05-09
9           1987
10          2003
11    2016-12-06
12          1973
13       1990-07
14    2004-01-01
15          1988
16    2012-09-25
17          2005
18    2010-09-23
19    2008-09-04
Name: published_date, dtype: object

In [7]:
# Vamos a usar regex para extraer cualquier grupo de 4 números consecutivos
df_temporal['year'] = df_temporal['published_date'].str.extract(r'(\d{4})')

In [8]:
# Comprobamos que no haya generado nulos 
df_temporal['year'].isnull().sum()

np.int64(0)

In [9]:
# Además, me gustaría tener el mes y tratándose de formato americano el mes siempre va a estar en medio. 
# Aplicamos de nuevo regex para ver si nos interesa quedarnos con ello o mejor descartar esta dimensión por la cantidad de ruido que genera (por ejemplo, demasiados nulos)
df_temporal['month'] = df_temporal['published_date'].str.extract(r'-(\d{2})')

In [10]:
df_temporal['month'].isnull().sum()

np.int64(12561)

In [11]:
df_temporal.shape

(40635, 11)

In [12]:
# En lugar de eliminarlos, les voy a imputar el valor 0 (como desconocido). 
df_temporal['month'] = df_temporal['month'].fillna(0).astype(int)

In [13]:
# Voy a añadir la columna month_name para agrupar con los nombres de los meses 
meses_dict = {
    0: 'Desconocido',
    1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 
    5: 'Mayo', 6: 'Junio', 7: 'Julio', 8: 'Agosto', 
    9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'
}

df_temporal['month_name'] = df_temporal['month'].map(meses_dict)

In [14]:
df_temporal['month_name'].value_counts()

month_name
Desconocido    12561
Enero           2995
Octubre         2768
Abril           2602
Septiembre      2569
Marzo           2433
Mayo            2320
Agosto          2215
Junio           2183
Febrero         2105
Diciembre       1996
Noviembre       1978
Julio           1910
Name: count, dtype: int64

#### La columna review/time: 

In [15]:
## En ese caso nos damos cuenta de que el código parece estar en Unix Timestamps, pero podemos pasarlo a datetime. 

In [16]:
rating_temporal.columns

Index(['id', 'title', 'price', 'user_id', 'profile_name', 'review/helpfulness',
       'review/score', 'review/time', 'review/summary', 'review/text'],
      dtype='object')

In [17]:
rating_temporal['review/time']

0          991440000
1         1291766400
2         1248307200
3         1222560000
4         1117065600
             ...    
414013    1285804800
414014    1230249600
414015    1179705600
414016    1111276800
414017    1045526400
Name: review/time, Length: 414018, dtype: int64

In [18]:
rating_temporal['review/time'] = pd.to_datetime(rating_temporal['review/time'], unit='s')
rating_temporal['review/time']

0        2001-06-02
1        2010-12-08
2        2009-07-23
3        2008-09-28
4        2005-05-26
            ...    
414013   2010-09-30
414014   2008-12-26
414015   2007-05-21
414016   2005-03-20
414017   2003-02-18
Name: review/time, Length: 414018, dtype: datetime64[ns]

In [19]:
# El día que se hicieron más reviews
rating_temporal['review/time'].value_counts()

review/time
2007-01-09    1037
2012-09-06     607
2013-02-18     561
2012-12-19     547
2012-12-28     547
              ... 
1997-02-07       1
1998-12-18       1
1997-11-22       1
1998-08-16       1
1997-09-06       1
Name: count, Length: 5685, dtype: int64

In [20]:
# Quiero separar años, de meses y días y crear columnas a parte para ver en qué año, qué mes y qué fecha suelen hacerse más reviews.
rating_temporal['year'] = rating_temporal['review/time'].dt.year
rating_temporal['month'] = rating_temporal['review/time'].dt.month
rating_temporal['day'] = rating_temporal['review/time'].dt.day
# Y vuelvo a crear una columna con los nombres de los meses
rating_temporal['month_name'] = rating_temporal['month'].map(meses_dict)

In [21]:
rating_temporal[['review/time', 'year', 'month',  'month_name', 'day']].head()

,review/time,year,month,month_name,day
0,2001-06-02,2001,6,Junio,2
1,2010-12-08,2010,12,Diciembre,8
2,2009-07-23,2009,7,Julio,23
3,2008-09-28,2008,9,Septiembre,28
4,2005-05-26,2005,5,Mayo,26


In [22]:
# El año de más reviews
rating_temporal['year'].value_counts()

year
2012    43696
2005    41502
2006    40380
2007    35012
2011    28873
2008    27906
2009    27737
2010    27192
2004    26600
2013    22542
2003    22452
2002    20696
2000    20256
2001    19292
1999     6237
1998     2998
1997      646
1996        1
Name: count, dtype: int64

In [23]:
# El mes de más reviews
rating_temporal['month_name'].value_counts()

month_name
Enero         48588
Febrero       40370
Diciembre     38395
Marzo         35031
Noviembre     32797
Agosto        32711
Octubre       31926
Septiembre    31916
Julio         31367
Mayo          31049
Abril         30294
Junio         29574
Name: count, dtype: int64

In [24]:
# El día de más reviews
rating_temporal['day'].value_counts()

day
9     15232
10    14410
6     14292
3     14030
8     14028
12    13938
7     13869
11    13836
28    13834
27    13792
5     13741
2     13588
26    13586
13    13564
21    13449
18    13444
19    13415
4     13386
23    13300
14    13266
20    13252
1     13249
22    13214
17    13185
16    13152
24    13132
25    13096
15    12873
30    12544
29    12416
31     7905
Name: count, dtype: int64

# RGB: Calculando sobre colores

In [25]:
df_temporal['color_rgb']

0         (164, 137, 90)
1        (145, 152, 157)
2        (200, 215, 214)
3        (158, 151, 134)
4           (62, 50, 50)
              ...       
40630    (241, 237, 235)
40631    (216, 212, 183)
40632     (103, 96, 156)
40633       (33, 28, 21)
40634    (249, 248, 229)
Name: color_rgb, Length: 40635, dtype: object

In [26]:
# Queremos agrupar por colores "primarios" los números de rgb
colores = {
    "Rojo": (255, 0, 0),
    "Verde": (0, 255, 0),
    "Azul": (0, 0, 255),
    "Amarillo": (255, 255, 0),
    "Blanco/Crema": (255, 255, 255),
    "Negro": (0, 0, 0),
    "Gris": (128, 128, 128),
    "Naranja": (255, 165, 0),
    "Morado": (128, 0, 128),
    "Rosa": (255, 192, 203),
    "Marrón": (139, 69, 19)
}

In [27]:
## Aplicamos ast porque la columna entiende que el rgb es string y no tupla 
df_temporal['color_rgb'] = df_temporal['color_rgb'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [28]:
## Aplicamos ahora la función y con el lambda hacemos que la guarde y la aplique a una columna una a una. 
df_temporal['color'] = df_temporal['color_rgb'].apply(lambda x: f.clasificar_color(x, colores))

In [29]:
df_temporal['color'].value_counts()

color
Gris            10642
Rosa            10314
Negro            7603
Marrón           6165
Blanco/Crema     2757
Naranja          1422
Morado            739
Rojo              521
Amarillo          339
Azul               95
Verde              36
Name: count, dtype: int64

# NLP: Calculando sentimientos

## Calculando la subjetividad

In [30]:
## Aplicamos otra función de NLP para que de los textos de review se calcule la subjetividad de cada texto. También lo aplicamos para descubrir la subjetividad de las sinopsis. 

In [31]:
rating_temporal

,id,title,price,user_id,profile_name,review/helpfulness,review/score,review/time,review/summary,review/text,year,month,day,month_name
0,0829814000,Wonderful Worship in Smaller Churches,19.40,AZ0IOBU20TBOP,Rev. Pamela Tinnin,8/10,5.0,2001-06-02,Outstanding Resource for Small Church Pastors,"I just finished the book, &quot;Wonderful Wors...",2001,6,2,Junio
1,0829814000,Wonderful Worship in Smaller Churches,19.40,A373VVEU6Z9M0N,Dr. Terry W. Dorsett,1/1,5.0,2010-12-08,Small Churches CAN Have Wonderful Worship,Many small churches feel like they can not hav...,2010,12,8,Diciembre
2,0829814000,Wonderful Worship in Smaller Churches,19.40,AGKGOH65VTRR4,"Cynthia L. Lajoy ""Cindy La Joy""",1/1,5.0,2009-07-23,Not Just for Pastors!,I just finished reading this amazing book and ...,2009,7,23,Julio
3,0829814000,Wonderful Worship in Smaller Churches,19.40,A3OQWLU31BU1Y,Maxwell Grant,1/1,5.0,2008-09-28,Small church pastor? This is the book on worship,I hadn't been a small church pastor very long ...,2008,9,28,Septiembre
4,0595344550,Whispers of the Wicked Saints,10.95,A3Q12RK71N74LB,Book Reader,7/11,1.0,2005-05-26,not good,I bought this book because I read some glowing...,2005,5,26,Mayo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
414013,0786182431,Very Bad Deaths: Library Edition,90.00,A1EC8SNPR56CLU,Denis Dube,0/0,4.0,2010-09-30,It's the way he writes it,"""Very Bad Death"" is a so so story, but the cha...",2010,9,30,Septiembre
414014,0786182431,Very Bad Deaths: Library Edition,90.00,A33VKWCAV9QQKC,"Paige E. Steadman ""RuneEnigma""",0/0,5.0,2008-12-26,"Bad Deaths, Great Book!",Very Bad Deaths was a very great book! Spider ...,2008,12,26,Diciembre
414015,0786182431,Very Bad Deaths: Library Edition,90.00,A2PK3NTC9RMEF4,S. Potter,0/0,3.0,2007-05-21,Still read it.,Anything by Spider Robinson is worth reading. ...,2007,5,21,Mayo
414016,0786182431,Very Bad Deaths: Library Edition,90.00,A2D0PY6HIGTYIT,"Adrian in Phoenix ""No Time for Fantasy""",5/8,5.0,2005-03-20,Not another Callahan story,Great novel! Easy & enjoyable to read straight...,2005,3,20,Marzo


In [32]:
# Quiero otorgarle una nota de subjetividad a cada uno de los rating text que tenemos en nuestro dataframe. 
rating_temporal['rating_subjectivity'] = rating_temporal['review/text'].apply(f.subjectivity)

In [33]:
# Nuestras estadísticas de subjetividad: 0 siendo objetivo y 1 siendo subjetivo. 
rating_temporal['rating_subjectivity'].describe()

count    414018.000000
mean          0.517766
std           0.141288
min           0.000000
25%           0.438889
50%           0.513166
75%           0.595683
max           1.000000
Name: rating_subjectivity, dtype: float64

In [34]:
# Quiero otorgarle una nota de subjetividad a cada uno de las descripciones de los libros que tenemos en nuestro dataframe. 
df_temporal['book_subjectivity'] = df_temporal['description'].apply(f.subjectivity)

In [35]:
# Nuestras estadísticas de subjetividad: 0 siendo objetivo y 1 siendo subjetivo. 
df_temporal['book_subjectivity'].describe()

count    40635.000000
mean         0.463159
std          0.192026
min          0.000000
25%          0.375000
50%          0.483333
75%          0.576136
max          1.000000
Name: book_subjectivity, dtype: float64

In [36]:
## Para conseguir qué palabras se usaron más, importamos un set de palabras que son comunmente usadas 
# pero que no aportan info, por ejemplo, and, the, a, of... 
## En ese set añadiremos las que tampoco nos aportan info en este caso como libro, read... 

## Calculando la polaridad

In [37]:
## Aplicamos otra función de NLP para que de los textos de review se calcule la polaridad de cada texto. También lo aplicamos para descubrir la polaridad de las sinopsis. 

In [38]:
# Quiero otorgarle una nota de subjetividad a cada uno de los rating text que tenemos en nuestro dataframe. 
rating_temporal['rating_polarity'] = rating_temporal['review/text'].apply(f.polarity) 

In [39]:
# Nuestras estadísticas de polaridad:-1 negativo, 0 siendo neutral y 1 positivo. 
rating_temporal['rating_polarity'].describe()

count    414018.000000
mean          0.217634
std           0.189913
min          -1.000000
25%           0.102778
50%           0.200000
75%           0.315909
max           1.000000
Name: rating_polarity, dtype: float64

In [40]:
# Quiero otorgarle una nota de subjetividad a cada uno de las descripciones de los libros que tenemos en nuestro dataframe. 
df_temporal['book_polarity'] = df_temporal['description'].apply(f.polarity)

In [41]:
# Nuestras estadísticas de polaridad:-1 negativo, 0 siendo neutral y 1 positivo. 
df_temporal['book_polarity'].describe()

count    40635.000000
mean         0.140500
std          0.174454
min         -1.000000
25%          0.033333
50%          0.138167
75%          0.238223
max          1.000000
Name: book_polarity, dtype: float64

## Calculando las palabras más usadas

In [42]:
## Python ofrece nltk una librería que nos permite descargar una lista de palabras comumente usadas en cualquier review de producto.
# A esa librería quiero añadirle palabras específicas para nuestro caso. 

In [43]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
# En este update he ido añadiendo palabras según los resultados para eliminar aquellas que siento que no aportan tanto al análisis. 
stop_words.update(['book', 'read', 'story', 'author', 'one', 
                   'like', 'many', 'even', 'character', 'plot', 
                   'well', 'time', 'publisher', 'description', 
                   'books', 'reader', 'readers', 'also', 'edition', 
                   'novel', 'make', 'find', 'would', 'much', 'really',
                   'people', 'found', 'dont', 'written', 'first', 'every',
                   'reading', 'think', 'still'])

[nltk_data] Downloading package stopwords to /Users/maria/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [44]:
## Hacemos el top de 20 palabras más usadas en las reviews, pero si hiciéramos un apply solo estaríamos localizando celda 
# por celda y no en todas las reviews, así que la pasamos la seri completa a la función para que 
# con un bucle recorra todas las descripciones y genere un conteo global.

In [45]:
sinopsis_top_20 = f.top_words(df_temporal['description'].dropna(), stop_words=stop_words, n=20)
sinopsis_top_20

[('life', 12472),
 ('world', 10145),
 ('years', 6261),
 ('history', 6152),
 ('love', 5877),
 ('work', 5349),
 ('family', 4904),
 ('american', 4769),
 ('times', 4728),
 ('york', 4517),
 ('young', 4237),
 ('best', 4137),
 ('great', 3664),
 ('series', 3618),
 ('stories', 3579),
 ('guide', 3528),
 ('help', 3451),
 ('including', 3351),
 ('lives', 3304),
 ('classic', 3235)]

In [46]:
## Hacemos lo mismo con las reviews
reviews_top_20 = f.top_words(rating_temporal['review/text'].dropna(), stop_words=stop_words, n=20)
reviews_top_20

[('good', 121520),
 ('great', 115750),
 ('life', 98039),
 ('love', 81247),
 ('know', 68336),
 ('work', 67048),
 ('could', 65420),
 ('characters', 59572),
 ('years', 59032),
 ('world', 57958),
 ('little', 57189),
 ('best', 52751),
 ('never', 50582),
 ('recommend', 50502),
 ('want', 50132),
 ('better', 46797),
 ('writing', 45241),
 ('things', 44826),
 ('history', 43914),
 ('information', 43439)]

## Calculando los temas más recurrentes

In [47]:
# Hacemos una lista para recoger todos los textos de las reviews y ometerls a Bertopic (modelo para analizar los temas pricipales)
reviews = rating_temporal['review/text'].tolist()

In [48]:
model = BERTopic(language="english", calculate_probabilities=False, verbose=True)

In [49]:
# Por un lado tenemos los topics (grupos de palabras clave) y probs (la matriz que dice cuánta probabilidad hay de que la review hable del topic que señalamos)
# Tarda cerca de media hora, así que lo pongo como comentario para solo ejecutarlo 1 vez. 
# topics, probs = model.fit_transform(reviews)

In [50]:
# Creamos una nueva columna en el dataset
rating_temporal['review/topic'] = topics

NameError: name 'topics' is not defined

In [ ]:
rating_temporal['review/topic'].max()

np.int64(4327)

In [ ]:
# El modelo le ha asociado un topic a cada una de las reviews. Ha calculado que hay 42329 topics únicos
# El -1 significa que de esos no ha encontrado tema. 
# Los diez más comentados son: 
top_topics = model.get_topic_info().head(12)
print(top_topics)

    Topic   Count                                      Name  \
0      -1  169875                         -1_my_our_it_this   
1       0    4334       0_condition_arrived_seller_shipping   
2       1    2112       1_kindle_version_formatting_edition   
3       2    1714  2_informative_reference_info_information   
4       3    1696           3_recipes_cookbook_cook_cooking   
5       4    1433                  4_poems_poetry_poem_poet   
6       5    1365                   5_bible_kjv_bibles_nasb   
7       6    1163                       6_her_she_shes_wait   
8       7     944        7_ending_plot_characters_storyline   
9       8     934        8_buddhism_buddhist_tibetan_buddha   
10      9     851           9_item_product_seller_condition   
11     10     843                10_loves_pictures_year_old   

                                       Representation  \
0   [my, our, it, this, author, read, book, we, me...   
1   [condition, arrived, seller, shipping, receive...   
2   [kind

In [ ]:
# Visualización de los 10 temas más recurrentes
model.visualize_barchart(top_n_topics=11)

In [ ]:
rating_temporal.groupby('review/topic')['rating_polarity'].mean().sort_values()

review/topic
2192   -0.299066
3213   -0.200899
4245   -0.181521
285    -0.097152
2673   -0.088108
          ...   
3755    0.617433
2515    0.637681
3882    0.652757
2311    0.687500
3666    0.786706
Name: rating_polarity, Length: 4329, dtype: float64

In [ ]:
topic_summary = rating_temporal.groupby('review/topic')['rating_polarity'].mean().reset_index()

In [ ]:
topic_summary['topic_name'] = topic_summary['review/topic'].apply(f.tema)

In [ ]:
odiados = topic_summary.sort_values(by='rating_polarity').head(5)
amados = topic_summary.sort_values(by='rating_polarity', ascending=False).head(5)

In [ ]:
print(odiados[['review/topic', 'topic_name', 'rating_polarity']])

      review/topic                         topic_name  rating_polarity
2193          2192  worst / mockingbirdquot / torture        -0.299066
3214          3213            boring / doodoo / sucks        -0.200899
4246          4245          yates / terrors / commits        -0.181521
286            285             boring / club / finish        -0.097152
2674          2673  seriously / informative / learned        -0.088108


In [ ]:
print(amados[['review/topic', 'topic_name', 'rating_polarity']])

      review/topic                             topic_name  rating_polarity
3667          3666     utmost / descriptive / wonderfully         0.786706
2312          2311                    pls / readso / soon         0.687500
3883          3882  anythingshe / carefullychosen / iswow         0.652757
2516          2515            bookshelf / digital / space         0.637681
3756          3755  exzillerateing / exsiting / storiesit         0.617433


# Limpieza final y unión entre ellos

In [ ]:
df_temporal.columns

Index(['title', 'description', 'authors', 'image', 'publisher',
       'published_date', 'categories', 'ratings_count', 'color_rgb', 'year',
       'month', 'month_name', 'color', 'book_subjectivity', 'book_polarity'],
      dtype='object')

In [ ]:
rating_temporal.columns

Index(['id', 'title', 'price', 'user_id', 'profile_name', 'review/helpfulness',
       'review/score', 'review/time', 'review/summary', 'review/text', 'year',
       'month', 'day', 'month_name', 'rating_subjectivity', 'rating_polarity',
       'review/topic'],
      dtype='object')

In [ ]:
# Teniendo en cuenta que queremos unir los dataframes, unifiquemos nombres de columnas

In [ ]:
books = df_temporal.rename(columns={'description': 'plot', 'image': 'cover',
                            'categories': 'genre', 'year': 'year_published',
                            'moth':'month_published'})

In [ ]:
reviews = rating_temporal.rename(columns={'id': 'isbn', 'year': 'year_reviewed', 
                                          'month':'month_reviewed', 'day':'day_reviewed',
                                          'rating_polarity':'review/polarity', 
                                          'rating_subjectivity': 'review/subjectivity'})

In [ ]:
books.columns

Index(['title', 'plot', 'authors', 'cover', 'publisher', 'published_date',
       'genre', 'ratings_count', 'color_rgb', 'year_published', 'month',
       'month_name', 'color', 'book_subjectivity', 'book_polarity'],
      dtype='object')

In [ ]:
books.shape

(40635, 15)

In [ ]:
reviews.columns

Index(['isbn', 'title', 'price', 'user_id', 'profile_name',
       'review/helpfulness', 'review/score', 'review/time', 'review/summary',
       'review/text', 'year_reviewed', 'month_reviewed', 'day_reviewed',
       'month_name', 'review/subjectivity', 'review/polarity', 'review/topic'],
      dtype='object')

In [ ]:
reviews.shape

(414018, 17)

In [ ]:
# Unimos los df
books_and_reviews = books.merge(reviews, on='title', how='inner')

In [ ]:
books_and_reviews.drop('month_name_x', axis=1)

,title,plot,authors,cover,publisher,published_date,genre,ratings_count,color_rgb,year_published,...,review/time,review/summary,review/text,year_reviewed,month_reviewed,day_reviewed,month_name_y,review/subjectivity,review/polarity,review/topic
0,The Church of Christ: A Biblical Ecclesiology ...,In The Church of Christ: A Biblical Ecclesiolo...,['Everett Ferguson'],http://books.google.com/books/content?id=kVqRa...,Wm. B. Eerdmans Publishing,1996,['Religion'],5.0,"(164, 137, 90)",1996,...,2000-04-11,Ecclesiological Milestone,With the publication of Everett Ferguson's boo...,2000,4,11,Abril,0.504196,0.180887,-1
1,The Church of Christ: A Biblical Ecclesiology ...,In The Church of Christ: A Biblical Ecclesiolo...,['Everett Ferguson'],http://books.google.com/books/content?id=kVqRa...,Wm. B. Eerdmans Publishing,1996,['Religion'],5.0,"(164, 137, 90)",1996,...,2011-07-24,Early Christian development of the Church,Everett Ferguson approaches the subject of ear...,2011,7,24,Julio,0.483222,0.050111,-1
2,The Church of Christ: A Biblical Ecclesiology ...,In The Church of Christ: A Biblical Ecclesiolo...,['Everett Ferguson'],http://books.google.com/books/content?id=kVqRa...,Wm. B. Eerdmans Publishing,1996,['Religion'],5.0,"(164, 137, 90)",1996,...,2010-11-17,An Excellent Presentation of the Beliefs of th...,This book is a continual resource. It is so bi...,2010,11,17,Noviembre,0.664286,0.425000,-1
3,The Church of Christ: A Biblical Ecclesiology ...,In The Church of Christ: A Biblical Ecclesiolo...,['Everett Ferguson'],http://books.google.com/books/content?id=kVqRa...,Wm. B. Eerdmans Publishing,1996,['Religion'],5.0,"(164, 137, 90)",1996,...,2010-02-15,Christ is Lord,This is a very useful and thorough text book. ...,2010,2,15,Febrero,0.050000,0.295000,1411
4,The Battleship Bismarck,The Bismarck is perhaps the most famous – and ...,['Stefan Draminski'],http://books.google.com/books/content?id=nxttD...,Bloomsbury Publishing,2018-09-20,['History'],1.0,"(200, 215, 214)",2018,...,2003-06-19,The Battleship Bismarck reviewed,This book is both a history and a photo album ...,2003,6,19,Junio,0.266667,0.135618,506
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
174079,The Orphan Of Ellis Island (Time Travel Advent...,"During a school trip to Ellis Island, Dominick...",['Elvira Woodruff'],http://books.google.com/books/content?id=J7M-N...,Scholastic Paperbacks,2000-06-01,['Juvenile Fiction'],2.0,"(33, 28, 21)",2000,...,2003-03-14,Good and sad book!!!!,This book from Elvira Wodruff is good. It tell...,2003,3,14,Marzo,0.398653,0.139099,-1
174080,The Autograph Man,Alex-Li Tandem sells autographs. His business ...,['Zadie Smith'],http://books.google.com/books/content?id=JM6YV...,Vintage,2003-08-12,['Fiction'],19.0,"(249, 248, 229)",2003,...,2005-10-16,very fractured,After reading 50 pages and restarting every so...,2005,10,16,Octubre,0.250000,0.250000,-1
174081,The Autograph Man,Alex-Li Tandem sells autographs. His business ...,['Zadie Smith'],http://books.google.com/books/content?id=JM6YV...,Vintage,2003-08-12,['Fiction'],19.0,"(249, 248, 229)",2003,...,2004-10-22,The quest for the holy Grail,What is wrong or good with this book. What are...,2004,10,22,Octubre,0.436954,0.088103,-1
174082,The Autograph Man,Alex-Li Tandem sells autographs. His business ...,['Zadie Smith'],http://books.google.com/books/content?id=JM6YV...,Vintage,2003-08-12,['Fiction'],19.0,"(249, 248, 229)",2003,...,2002-11-11,Looking For Spirituality in All The Wrong Places,Zadie Smith's latest is a subpar Coupland-ish ...,2002,11,11,Noviembre,0.488889,0.265741,563


In [ ]:
books_and_reviews.rename(columns={'month_name_y':'month_name'})

,title,plot,authors,cover,publisher,published_date,genre,ratings_count,color_rgb,year_published,...,review/time,review/summary,review/text,year_reviewed,month_reviewed,day_reviewed,month_name,review/subjectivity,review/polarity,review/topic
0,The Church of Christ: A Biblical Ecclesiology ...,In The Church of Christ: A Biblical Ecclesiolo...,['Everett Ferguson'],http://books.google.com/books/content?id=kVqRa...,Wm. B. Eerdmans Publishing,1996,['Religion'],5.0,"(164, 137, 90)",1996,...,2000-04-11,Ecclesiological Milestone,With the publication of Everett Ferguson's boo...,2000,4,11,Abril,0.504196,0.180887,-1
1,The Church of Christ: A Biblical Ecclesiology ...,In The Church of Christ: A Biblical Ecclesiolo...,['Everett Ferguson'],http://books.google.com/books/content?id=kVqRa...,Wm. B. Eerdmans Publishing,1996,['Religion'],5.0,"(164, 137, 90)",1996,...,2011-07-24,Early Christian development of the Church,Everett Ferguson approaches the subject of ear...,2011,7,24,Julio,0.483222,0.050111,-1
2,The Church of Christ: A Biblical Ecclesiology ...,In The Church of Christ: A Biblical Ecclesiolo...,['Everett Ferguson'],http://books.google.com/books/content?id=kVqRa...,Wm. B. Eerdmans Publishing,1996,['Religion'],5.0,"(164, 137, 90)",1996,...,2010-11-17,An Excellent Presentation of the Beliefs of th...,This book is a continual resource. It is so bi...,2010,11,17,Noviembre,0.664286,0.425000,-1
3,The Church of Christ: A Biblical Ecclesiology ...,In The Church of Christ: A Biblical Ecclesiolo...,['Everett Ferguson'],http://books.google.com/books/content?id=kVqRa...,Wm. B. Eerdmans Publishing,1996,['Religion'],5.0,"(164, 137, 90)",1996,...,2010-02-15,Christ is Lord,This is a very useful and thorough text book. ...,2010,2,15,Febrero,0.050000,0.295000,1411
4,The Battleship Bismarck,The Bismarck is perhaps the most famous – and ...,['Stefan Draminski'],http://books.google.com/books/content?id=nxttD...,Bloomsbury Publishing,2018-09-20,['History'],1.0,"(200, 215, 214)",2018,...,2003-06-19,The Battleship Bismarck reviewed,This book is both a history and a photo album ...,2003,6,19,Junio,0.266667,0.135618,506
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
174079,The Orphan Of Ellis Island (Time Travel Advent...,"During a school trip to Ellis Island, Dominick...",['Elvira Woodruff'],http://books.google.com/books/content?id=J7M-N...,Scholastic Paperbacks,2000-06-01,['Juvenile Fiction'],2.0,"(33, 28, 21)",2000,...,2003-03-14,Good and sad book!!!!,This book from Elvira Wodruff is good. It tell...,2003,3,14,Marzo,0.398653,0.139099,-1
174080,The Autograph Man,Alex-Li Tandem sells autographs. His business ...,['Zadie Smith'],http://books.google.com/books/content?id=JM6YV...,Vintage,2003-08-12,['Fiction'],19.0,"(249, 248, 229)",2003,...,2005-10-16,very fractured,After reading 50 pages and restarting every so...,2005,10,16,Octubre,0.250000,0.250000,-1
174081,The Autograph Man,Alex-Li Tandem sells autographs. His business ...,['Zadie Smith'],http://books.google.com/books/content?id=JM6YV...,Vintage,2003-08-12,['Fiction'],19.0,"(249, 248, 229)",2003,...,2004-10-22,The quest for the holy Grail,What is wrong or good with this book. What are...,2004,10,22,Octubre,0.436954,0.088103,-1
174082,The Autograph Man,Alex-Li Tandem sells autographs. His business ...,['Zadie Smith'],http://books.google.com/books/content?id=JM6YV...,Vintage,2003-08-12,['Fiction'],19.0,"(249, 248, 229)",2003,...,2002-11-11,Looking For Spirituality in All The Wrong Places,Zadie Smith's latest is a subpar Coupland-ish ...,2002,11,11,Noviembre,0.488889,0.265741,563


In [ ]:
# Guardamos el df para el siguiene paso
books_and_reviews.to_csv('data/books_and_reviews.csv', index=False)

# Diccionario final

1. Info libro
`title`: Título del libro
`authors`: Autor o autores del libro
`plot`: Resumen o sinopsis de la trama
`genre`: Género literario
`publisher`: Editorial
`published_date`: Fecha de publicación original
`isbn`: Código identificador internacional (ISBN)
`price`: Precio del libro
`ratings_count`: Cantidad total de calificaciones del libro

2. Atributos visuales
`cover`: Enlace a la imagen de portada
`color`: Color predominante identificado
`color_rgb`: Código de color en formato RGB

3. Análisis de sentimiento (Libro)
`book_subjectivity`: Nivel de subjetividad del texto del libro
`book_polarity`: Sentimiento (positivo/negativo) del contenido del libro

4. Datos del usuario y reseña
`user_id`: Identificador único del usuario
`profile_name`: Nombre de perfil del usuario
`review/score`: Puntuación otorgada (ej. 1 a 5 estrellas) a la review
`review/summary`: Título o resumen corto de la reseña
`review/text`: Cuerpo completo de la opinión del usuario
`review/helpfulness`: Ratio de utilidad de la reseña
`review/topic`: Tema principal detectado en la reseña

5. Fechas y temporalidad
`year_published`: Año de publicación
`month`: Mes de publicación
`month_name`: Nombre del mes de publicación
`year_reviewed`: Año en que se escribió la reseña
`month_reviewed`: Mes de la reseña
`day_reviewed`: Día del mes de la reseña

6. Análisis de sentimiento (review)
`review/subjectivity`: Qué tan subjetiva es la opinión del usuario
`review/polarity`: Qué tan positiva o negativa es la reseña